# Two branch architecture
- with images generated as deformed

In [1]:
%load_ext autoreload
%autoreload 2
from constraints.models.deform_only import TwoBranch
from constraints.generators import ArteryGeneratorDeformed
from constraints.visu import show_torch_image
from constraints import REPO_ROOT
import pytorch_lightning as pl
import neurite as ne
import torch

/mnt/appl/software/protobuf-python/6.31.1-GCCcore-14.2.0/lib/python3.13/site-packages/google/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [2]:
print(REPO_ROOT)

/mnt/personal/mrkosmic/synced/constraints


In [25]:
dataset_trn = ArteryGeneratorDeformed(num_samples=1000, fixed_seed=42, magnitude=4.0, integrations=2, scales=15,fractal_mode="upsample")
dataset_val = ArteryGeneratorDeformed(num_samples=100, fixed_seed=43, magnitude=4.0, integrations=3, scales=15, fractal_mode="upsample")
for i in range(1):
    trn1 = dataset_trn[i]
    for k in trn1.keys():
        
        print(k, trn1[k].shape)
        if k == "field": continue
        show_torch_image(trn1[k],  cmap="gray" if k == "img" else None)
    

img torch.Size([1, 256, 256])
mask torch.Size([3, 256, 256])
template torch.Size([3, 256, 256])
field torch.Size([2, 256, 256])


In [ ]:
class LitTwoBranch(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = TwoBranch(
            source_channels=1,
            target_channels=3,
            nb_features=[32, 32, 32, 32],
            integration_steps=5,
        )
        
        self.template_loss_fn =torch.nn.MSELoss() #TODO centroid or SDF loss
        
        self.grad_loss_fn = ne.nn.modules.SpatialGradient('l2')
        self.grad_loss_weight = 0.05
        self.segmentation_loss_fn = torch.nn.CrossEntropyLoss()
        
    def forward(self, img: torch.Tensor, template: torch.Tensor|None = None):
        return self.model(img, template)
    
    def _shared_step(self,batch,stage):
        template = batch["template"]
        img = batch["img"]
        mask = batch["mask"]
        segmentation_logits,field, warped_template = self(img, template)
        template_loss = self.template_loss_fn(warped_template, mask)
        grad_loss = self.grad_loss_fn(field)
        segmentation_loss = self.segmentation_loss_fn(segmentation_logits, mask)
        # keep template_loss now for debugging
        loss = template_loss + self.grad_loss_weight * grad_loss + segmentation_loss
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_template_loss", template_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_grad_loss", grad_loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_segmentation_loss", segmentation_loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss        
        
    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')
    
    
    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        template = batch["template"]
        mask = batch["mask"]
        img = batch["img"]
        segmentation_logits,field, warped_template = self(img, template)
        return {
            "segmentation_logits": segmentation_logits,
            "field": field,
            "warped_template": warped_template,
            "mask": mask,
            "img": img
        }
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-1)
        return optimizer

In [32]:
EPOCHS= 100
BATCH_SIZE = 4
from pytorch_lightning.loggers import CSVLogger
from pathlib import Path

logger = CSVLogger('logs', name='two_branch_deformation')
model = LitTwoBranch()

# overfit one batch
trn_loader = torch.utils.data.DataLoader(dataset_trn, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices='auto',
    logger=logger,
    overfit_batches=1,
)
trainer.fit(model, trn_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                 | Type             | Params | Mode 
------------------------------------------------------------------
0 | model                | TwoBranch        | 11.6 M | train
1 | template_loss_fn     | MSELoss          | 0      | train
2 | grad_loss_fn         | SpatialGradient  | 0      | train
3 | segmentation_loss_fn | CrossEntropyLoss | 0      | train
------------------------------------------------------------------
11.6 M    Trainable params
0         Non-trainable params
11.6 M    Total params
46.387    Total estimated model params size (MB

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [33]:
#save the overfitted model predictions
batch = next(iter(trn_loader))
res = model.predict_step(batch,0)


In [34]:
print(str(Path(".").resolve()))
print(list(Path(".").iterdir()))

/mnt/personal/mrkosmic/synced/constraints
[PosixPath('mutagen.yml.lock'), PosixPath('README.md'), PosixPath('remote_submit.sh'), PosixPath('pyproject.toml'), PosixPath('.python-version'), PosixPath('scripts'), PosixPath('uv.lock'), PosixPath('.gitignore'), PosixPath('.ruff_cache'), PosixPath('notebooks'), PosixPath('.venv'), PosixPath('submit_job.sh'), PosixPath('mnist-data'), PosixPath('outputs'), PosixPath('lightning_logs'), PosixPath('main.py'), PosixPath('tests'), PosixPath('start_jupyter.sh'), PosixPath('experiments'), PosixPath('remote_jupyter.sh'), PosixPath('logs'), PosixPath('slurm'), PosixPath('constraints'), PosixPath('mutagen.yml'), PosixPath('load_env.sh')]


In [35]:
print(res.keys())
output_dir = Path("./notebooks/ex2/overfit_output")
output_dir.mkdir(parents=True, exist_ok=True)
warped_templates = res["warped_template"]
masks = res["mask"]
print(warped_templates.shape, masks.shape)
for i in range(warped_templates.shape[0]):
    show_torch_image(warped_templates[i],save_path=output_dir / f"warped_template_{i}.png")
    show_torch_image(masks[i],save_path=output_dir / f"mask_{i}.png")

dict_keys(['segmentation_logits', 'field', 'warped_template', 'mask', 'img'])
torch.Size([4, 3, 256, 256]) torch.Size([4, 3, 256, 256])
